# 050 — Data Augmentation

Manufacture training data from the data you already have. Measured on 300
digit images: test accuracy **91.1% → 93.0%**, with the train/test gap
narrowing from **8.9%** to **7.0%**.

Needs `scikit-learn` and `numpy` only.

| Part | What we check |
|---|---|
| A | the baseline on 300 images |
| B | four one-pixel shifts, 1,500 images, and what it buys |
| C | augmentation enlarges AND closes the gap |
| D | the rule with no exceptions: never augment the test set |

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Part A — The baseline

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

digits = load_digits()
X_tr, X_te, y_tr, y_te = train_test_split(
    digits.images, digits.target, train_size=300, random_state=0,
    stratify=digits.target)

def evaluate(X, y, label):
    clf = MLPClassifier(hidden_layer_sizes=(64,), max_iter=1500, random_state=0)
    clf.fit(X.reshape(len(X), -1), y)
    train = clf.score(X.reshape(len(X), -1), y)
    test = clf.score(X_te.reshape(len(X_te), -1), y_te)
    print(f"{label:<32} train {train:.1%}  test {test:.1%}  gap {train - test:+.1%}")
    return train, test

base_train, base_test = evaluate(X_tr, y_tr, f"original ({len(X_tr)} images)")
assert round(base_test * 100, 1) == 91.1

## Part B — The crudest possible augmentation

Four one-pixel shifts: up, down, left, right. No rotation, no zoom, no flip.

In [ ]:
def shift(images, dy, dx):
    return np.array([np.roll(np.roll(im, dy, axis=0), dx, axis=1) for im in images])

shifts = [(1, 0), (-1, 0), (0, 1), (0, -1)]
X_aug = np.concatenate([X_tr] + [shift(X_tr, dy, dx) for dy, dx in shifts])
y_aug = np.tile(y_tr, 1 + len(shifts))
print(f"{len(X_tr)} images became {len(X_aug)}")
assert len(X_aug) == 1500 and len(y_aug) == 1500

aug_train, aug_test = evaluate(X_aug, y_aug, f"+ shifts ({len(X_aug)} images)")
assert round(aug_test * 100, 1) == 93.0
print(f"\ntest accuracy {base_test:.1%} -> {aug_test:.1%}  ({aug_test - base_test:+.1%})")

## Part C — Two effects, one experiment

In [ ]:
base_gap, aug_gap = base_train - base_test, aug_train - aug_test
print(f"gap before {base_gap:+.1%}   after {aug_gap:+.1%}")
assert round(base_gap * 100, 1) == 8.9
assert round(aug_gap * 100, 1) == 7.0
assert aug_test > base_test and aug_gap < base_gap
print("the dataset got 5x bigger AND the generalisation gap narrowed")
print("note the modesty of the gain: +1.9% from the crudest augmentation there is")

## Part D — Never augment what you measure on

Augmenting the test set changes the question you are answering. Here is what it
does to the number you would report.

In [ ]:
clf = MLPClassifier(hidden_layer_sizes=(64,), max_iter=1500, random_state=0)
clf.fit(X_aug.reshape(len(X_aug), -1), y_aug)

honest = clf.score(X_te.reshape(len(X_te), -1), y_te)
X_te_aug = np.concatenate([X_te] + [shift(X_te, dy, dx) for dy, dx in shifts])
y_te_aug = np.tile(y_te, 1 + len(shifts))
inflated = clf.score(X_te_aug.reshape(len(X_te_aug), -1), y_te_aug)

print(f"honest test set        {honest:.1%}  ({len(X_te)} images)")
print(f"augmented test set     {inflated:.1%}  ({len(X_te_aug)} images)")
print(f"difference             {inflated - honest:+.1%}")
print()
print("Either way the number is no longer about unseen data: every shifted copy")
print("of a test image shares its original. Augment training data only.")

## What this notebook established

- 300 images, four one-pixel shifts: test accuracy **91.1% → 93.0%**.
- The train/test gap narrowed from **8.9%** to **7.0%** — augmentation enlarges
  the dataset and attacks overfitting in one move.
- The gain is real but modest. Augmentation is not a substitute for data.
- Never augment validation or test data.

## Exercises

1. Add 15-degree rotations with `scipy.ndimage.rotate`. Does the gain grow?
2. Try `train_size=1200` instead of 300. Where does augmentation stop helping?
3. Add a vertical flip. Explain the result — digits are not flip-invariant.